# Novel View Synthesis

A radiance field represents a scene as density and colour at every point, from
which volume rendering draws any view. This notebook compares backbones for the
density, on the Blender scene lego, under FINER's protocol in a quick pass: 25
training views at 200 × 200 pixels, as
[torch-ngp](https://github.com/ashawkey/torch-ngp) runs it with
`--downscale 4 --trainskip 4 --bound 1 --scale 0.8 --dt_gamma 0`, and 5000
steps of 4096 rays instead of the benchmark's 25,000 (torch-ngp's `--iter` is
37,500), scored
on all 200 test views. Every backbone shares the colour network and the
renderer, so the spatial representation is the only variable. The backbones
are FUTON, with a sinc and a cosine basis, and five representative baselines
of the benchmark: RFF (Fourier features), SIREN and FINER (periodic
activations), Instant-NGP (a hash grid) and TensoRF (a grid factorization, in
its VM form). Every setting is that of `configs/nerf.yaml`, which holds every
model of the benchmark.

## Setup

In [ ]:
import copy
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from matplotlib.ticker import (
    FixedLocator,
    MaxNLocator,
    MultipleLocator,
    NullFormatter,
    StrMethodFormatter,
)
from matplotlib_inline.backend_inline import set_matplotlib_formats
import imageio.v2 as imageio
from IPython.display import Image as IPyImage, display

import neurofield as nf

set_matplotlib_formats("retina")  # sharp figures on high-resolution screens
sns.set_theme(
    context="notebook", style="whitegrid", font_scale=0.85, rc={"grid.color": ".92"}
)

colorblind = sns.color_palette("colorblind")
# FUTON in black, its cosine basis dashed; the periodic activations in warm
# hues and the grids in cool ones.
PALETTE = {
    "FUTON (sinc)": "black",
    "FUTON (cosine)": "black",
    "RFF": colorblind[4],
    "SIREN": colorblind[1],
    "FINER": colorblind[3],
    "Instant-NGP": colorblind[2],
    "TensoRF": colorblind[0],
}
DASHES = {model: (4, 2) if model == "FUTON (cosine)" else "" for model in PALETTE}
# seaborn's own markers, each fixed to a model as its colour is
MARKERS = dict(zip(PALETTE, ["o", "X", (4, 0, 45), "P", (4, 0, 0), (4, 1, 0), "^"]))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backend = "nerfacc" if nf.nerf.is_nerfacc_available() else "pytorch"
print(f"Device: {device}, renderer: {backend}")

The plotting helpers: training time on a log axis, the metric on whole
decibels or half points, a summary table, images side by side, and
reconstructions above their errors.

In [ ]:
NICE = [1, 2, 5, 10]  # tick steps: 1, 2 or 5 times a power of ten


def time_axis(ax, times):
    """Training time on a log axis, ticked 1, 2, 5, 10, ... in plain numbers."""
    low, high = times.min() / 1.1, times.max() * 1.1
    ax.set(xscale="log", xlim=(low, high), xlabel="Training time (s)")
    for subs in ((1, 2, 5), (1,)):
        ticks = [m * 10.0**k for k in range(-3, 6) for m in subs]
        ticks = [tick for tick in ticks if low <= tick <= high]
        if len(ticks) <= 7:
            break
    ax.xaxis.set_major_locator(FixedLocator(ticks))
    ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}"))
    ax.xaxis.set_minor_formatter(NullFormatter())


def metric_axis(ax, metric, low, high):
    """The metric's axis over [low, high]: whole decibels for a PSNR, half
    points for an IoU, which cannot pass 100."""
    if metric == "IoU (%)":
        low = np.floor(2 * low) / 2
        ax.set_ylim(low, min(max(np.ceil(2 * high) / 2, low + 0.5), 100))
        ax.yaxis.set_major_locator(MultipleLocator(0.5))
    else:
        low = np.floor(low)
        ax.set_ylim(low, max(np.ceil(high), low + 1))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6, steps=NICE, integer=True))


def table(summary, formats, better):
    """The summary, each column in its format and, in each column of
    ``better`` ("higher" or "lower"), the best value bold and the second
    underlined."""

    def marks(column):
        rank = column.rank(method="min", ascending=better[column.name] == "lower")
        looks = {1: "font-weight: bold", 2: "text-decoration: underline"}
        return [looks.get(r, "") for r in rank]

    return (
        summary.style.format(formats, na_rep="")
        .apply(marks, subset=list(better))
        .hide(axis="index")
    )


def show(panels, ncols, width=11, **imshow):
    """Images side by side under their titles, in rows of ``ncols``, each
    drawn at the size of the first."""
    height, side = panels[0][1].shape[:2]
    nrows = -(-len(panels) // ncols)
    room = 0.45 if any("\n" in title for title, _ in panels) else 0.3
    fig, axes = plt.subplots(
        nrows,
        ncols,
        squeeze=False,
        layout="constrained",
        figsize=(width, nrows * (width / ncols * height / side + room)),
    )
    for ax in axes.flat:
        ax.axis("off")
    for ax, (title, image) in zip(axes.flat, panels):
        ax.imshow(image, interpolation="nearest", **imshow)
        ax.set_title(title, fontsize=9)
    return fig


def compare(panels, label, width=11):
    """The reference and the reconstructions in a row under their titles
    (``panels``, the reference first), and beneath each reconstruction its
    absolute error, averaged over the colour channels; the errors share one
    scale, which saturates at their 99th percentile, and its colour bar
    stands beneath the reference."""
    reference = panels[0][1].astype(float)
    errors = [np.abs(image - reference).mean(-1) for _, image in panels[1:]]
    height, side = reference.shape[:2]
    ncols = len(panels)
    fig, axes = plt.subplots(
        2,
        ncols,
        layout="constrained",
        figsize=(width, 2 * width / ncols * height / side + 0.45),
    )
    for ax in axes.flat:
        ax.axis("off")
    for ax, (title, image) in zip(axes[0], panels):
        ax.imshow(image, interpolation="nearest")
        ax.set_title(title, fontsize=9)
    vmax = np.percentile(np.concatenate([error.ravel() for error in errors]), 99)
    for ax, error in zip(axes[1, 1:], errors):
        maps = ax.imshow(
            error, cmap="magma", vmin=0, vmax=vmax, interpolation="nearest"
        )
    bar = fig.colorbar(maps, cax=axes[1, 0].inset_axes([0.8, 0.04, 0.07, 0.92]))
    bar.ax.yaxis.set_ticks_position("left")
    bar.ax.yaxis.set_label_position("left")
    bar.ax.yaxis.set_major_locator(MaxNLocator(nbins=4, steps=NICE))
    bar.ax.tick_params(labelsize=8)
    bar.set_label(label, fontsize=9)
    bar.outline.set_visible(False)
    return fig

## Data

Four of the 25 training views.

In [ ]:
SCENE = "lego"
DATA_PATH = f"../data/nerf/blender/{SCENE}"
DOWNSAMPLE = 4  # 200 x 200 pixels (torch-ngp --downscale)
TRAIN_SKIP = 4  # 25 of the 100 training views (torch-ngp --trainskip)

train_dataset = nf.nerf.BlenderDataset(
    DATA_PATH, split="train", downsample=DOWNSAMPLE, skip=TRAIN_SKIP
)
val_dataset = nf.nerf.BlenderDataset(DATA_PATH, split="val", downsample=DOWNSAMPLE)
test_dataset = nf.nerf.BlenderDataset(DATA_PATH, split="test", downsample=DOWNSAMPLE)
print(
    f"Views: {len(train_dataset)} training, {len(val_dataset)} validation, "
    f"{len(test_dataset)} test, at {train_dataset.height} x {train_dataset.width}"
)

views = np.linspace(0, len(train_dataset) - 1, 4, dtype=int)
show(
    [
        (f"View {i}", train_dataset.composite(train_dataset.images[i]).cpu().numpy())
        for i in views
    ],
    ncols=4,
)
plt.show()

## Models

Each entry gives a backbone's class, its arguments and its learning rate. A
backbone outputs the density and 15 geometry features, which a colour network
shared by every field turns into colour; each field holds about 75k parameters.
`# deviation` marks a setting that departs from the authors' code, with the
reason.

In [ ]:
# Shared configuration across all models
GEOMETRY_FEATURES = 15
COLOR_NET = ("mlp", {"hidden_features": 64, "hidden_layers": 2})

# Model-specific configurations
CONFIGS = [
    {
        "name": "RFF",
        "density_net": (
            nf.RFF,
            {
                "hidden_features": 180,
                "hidden_layers": 2,
                "num_frequencies": 90,
                "sigma": 2.0,  # tuned on drums and lego: 8 loses 4 dB
            },
        ),
        "lr": 3e-3,
    },
    {
        "name": "SIREN",
        "density_net": (
            nf.SIREN,
            {"hidden_features": 180, "hidden_layers": 3, "omega": 30.0},
        ),
        "lr": 3e-4,
    },
    {
        "name": "FINER",
        "density_net": (
            nf.FINER,
            {"hidden_features": 180, "hidden_layers": 3, "omega": 30.0},
        ),
        "lr": 3e-4,
    },
    {
        "name": "Instant-NGP",  # instant-ngp density network, paper hash grid
        "density_net": (
            nf.InstantNGP,
            {
                "num_levels": 16,
                "features_per_level": 2,
                "log2_hashmap_size": 11,
                "base_resolution": 16,
                "max_resolution": 2048,
                "hidden_features": 64,
                "hidden_layers": 1,
            },
        ),
        # deviation: torch-ngp's 1e-2 costs 1.3 dB on lego's validation views
        "lr": 3e-2,
    },
    {
        "name": "TensoRF",
        "density_net": (
            nf.TensoRF,
            {
                "rank": 14,
                "resolution": 32,
                "mode": "vm",
                "hidden_features": 128,
                "hidden_layers": 2,
            },
        ),
        "lr": 3e-2,
    },
    {
        "name": "FUTON (sinc)",
        "density_net": (
            nf.FUTON,
            {
                "basis": ("sinc", {"num_components": 128}),
                "combiner": ("cp", {"rank": 150}),
                "decoder": ("mlp", {"hidden_layers": 1, "hidden_features": 64}),
            },
        ),
        "lr": 1e-2,
    },
    {
        "name": "FUTON (cosine)",
        "density_net": (
            nf.FUTON,
            {
                "basis": ("cosine", {"num_components": 128}),
                "combiner": ("cp", {"rank": 150}),
                "decoder": ("mlp", {"hidden_layers": 1, "hidden_features": 64}),
            },
        ),
        "lr": 1e-2,
    },
]

## Training

The density and colour networks are compiled with `torch.compile`, with
dynamic shapes, since the samples a batch of rays keeps vary from step to step.
An untimed occupancy update and rendered batch compile them before training
starts, on a copy of the renderer, so that the training times exclude
compilation and the renderer that trains starts from an empty grid. Every 500
steps the field renders four validation views, which the times exclude too.
Set `NUM_STEPS = 25_000` for the benchmark's full schedule.

In [ ]:
NUM_STEPS = 5_000  # a quick pass; the benchmark trains for 25_000
NUM_RAYS = 4_096  # torch-ngp --num_rays
MAX_SAMPLES_PER_RAY = 1_024  # torch-ngp --max_steps
EVAL_VIEWS = list(range(0, len(val_dataset), 25))  # 4 validation views
results = {}

for config in CONFIGS:
    torch.manual_seed(0)
    field = nf.nerf.RadianceField(
        density_net=config["density_net"],
        color_net=COLOR_NET,
        geometry_features=GEOMETRY_FEATURES,
        direction_encoding="sh",
        aabb=train_dataset.aabb,
    )
    renderer = nf.nerf.create_renderer(
        backend=backend,
        aabb=train_dataset.aabb,
        near=train_dataset.near,
        far=train_dataset.far,
    )
    print(f"{config['name']}: {nf.count_parameters(field):,} parameters")
    for network in (field.density_net, field.color_net):
        network.compile(dynamic=True)
    warm_up = copy.deepcopy(renderer).to(device)
    warm_up.update_occupancy(field.to(device), 0)
    view = train_dataset[0]
    rays_o = view["rays_o"].reshape(-1, 3)[:NUM_RAYS].to(device)
    rays_d = view["rays_d"].reshape(-1, 3)[:NUM_RAYS].to(device)
    warm_up(field, rays_o, rays_d)["rgb"].sum().backward()  # compiles them
    field.zero_grad(set_to_none=True)
    result = nf.nerf.train(
        field,
        renderer,
        train_dataset,
        val_dataset,
        num_steps=NUM_STEPS,
        num_rays=NUM_RAYS,
        max_samples_per_ray=MAX_SAMPLES_PER_RAY,
        lr=config["lr"],
        eval_interval=500,
        eval_indices=EVAL_VIEWS,
        log_interval=100,
        device=device,
    )
    result["field"], result["renderer"] = field, renderer
    results[config["name"]] = result

    torch.cuda.empty_cache()
    torch.compiler.reset()  # the next model compiles anew

## Convergence

The mean PSNR of the four validation views, every 500 steps.

FUTON is black, its cosine basis dashed; the periodic activations have warm
hues and the grids cool ones.

In [ ]:
records = pd.DataFrame(
    {
        "Model": name,
        "Step": entry["step"],
        "Time (s)": entry["elapsed"],
        "PSNR (dB)": entry["eval"]["psnr"],
    }
    for name, result in results.items()
    for entry in result["history"]
    if "eval" in entry
)

order = [model for model in PALETTE if model in set(records["Model"])]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True, layout="constrained")
for ax, x in zip(axes, ("Step", "Time (s)")):
    sns.lineplot(
        records,
        x=x,
        y="PSNR (dB)",
        hue="Model",
        style="Model",
        hue_order=order,
        style_order=order,
        palette=PALETTE,
        dashes=DASHES,
        legend=ax is axes[1],
        ax=ax,
    )
axes[0].xaxis.set_major_locator(MaxNLocator(nbins=5, steps=NICE))
axes[0].xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
final = records.groupby("Model")["PSNR (dB)"].last()
low, high = final.min(), records["PSNR (dB)"].max()
margin = 0.05 * (high - low)  # so that no curve ends on the frame
metric_axis(axes[0], "PSNR (dB)", low - margin, high + margin)
shown = records["PSNR (dB)"] >= axes[0].get_ylim()[0]  # the evaluations in view
time_axis(axes[1], records.loc[shown, "Time (s)"])

sns.move_legend(
    axes[-1], "upper left", bbox_to_anchor=(1.02, 1), title=None, frameon=False
)
sns.despine(fig, offset=3)
plt.show()

## Summary

Every field renders all 200 test views. The rendering speed is the median rate
of those renders, the first of which compiles the networks anew, and inference
memory their median peak; training memory is the peak of training. In each
column the best value is bold and the second underlined.

In [ ]:
rows = []
for name, result in results.items():
    torch.compiler.reset()  # each field compiles alone, as in training
    test = nf.nerf.evaluate(result["field"], result["renderer"], test_dataset)
    result["test"], views = test, test["views"]
    rows.append(
        {
            "Model": name,
            "Params (k)": result["config"]["num_params"] / 1e3,
            "Training time (s)": result["history"][-1]["elapsed"],
            "Training memory (MB)": result["config"]["peak_memory"] / 1e6,
            "Rendering speed (FPS)": 1 / np.median([v["duration"] for v in views]),
            "Inference memory (MB)": np.median([v["peak_memory"] for v in views]) / 1e6,
            "PSNR (dB)": test["mean"]["psnr"],
            "SSIM": test["mean"]["ssim"],
            "LPIPS": test["mean"]["lpips"],
        }
    )
summary = pd.DataFrame(rows).sort_values("PSNR (dB)", ascending=False)

formats = {
    "Params (k)": "{:.1f}",
    "Training time (s)": "{:.1f}",
    "Training memory (MB)": "{:.0f}",
    "Rendering speed (FPS)": "{:.1f}",
    "Inference memory (MB)": "{:.0f}",
    "PSNR (dB)": "{:.2f}",
    "SSIM": "{:.3f}",
    "LPIPS": "{:.3f}",
}
better = {
    "Training time (s)": "lower",
    "Training memory (MB)": "lower",
    "Rendering speed (FPS)": "higher",
    "Inference memory (MB)": "lower",
    "PSNR (dB)": "higher",
    "SSIM": "higher",
    "LPIPS": "lower",
}
table(summary, formats, better)

In [ ]:
order = [model for model in PALETTE if model in set(summary["Model"])]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True, layout="constrained")
for ax, cost in zip(axes, ("Training time (s)", "Rendering speed (FPS)")):
    sns.scatterplot(
        summary,
        x=cost,
        y="PSNR (dB)",
        hue="Model",
        style="Model",
        hue_order=order,
        style_order=order,
        palette=PALETTE,
        markers=MARKERS,
        s=70,
        legend=ax is axes[1],
        ax=ax,
    )
    ax.margins(x=0.08)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5, steps=NICE))
axes[0].set_xlabel("Training time (s) ↓")
axes[1].set_xlabel("Rendering speed (FPS) ↑")
low, high = summary["PSNR (dB)"].min(), summary["PSNR (dB)"].max()
metric_axis(axes[0], "PSNR (dB)", low - 0.08 * (high - low), high + 0.08 * (high - low))

sns.move_legend(
    axes[-1], "upper left", bbox_to_anchor=(1.02, 1), title=None, frameon=False
)
sns.despine(fig, offset=3)
plt.show()

## Qualitative Comparison

A test view in every field, titled with the view's PSNR, and beneath, the error
of each, averaged over the colour channels, on one scale.

In [ ]:
VIEW = 55  # of the test split
batch = test_dataset[VIEW]
target = test_dataset.composite(batch["rgba"]).cpu()
renders, scores = {}, {}
for name in summary["Model"]:
    with torch.compiler.set_stance("force_eager"):  # a figure needs no speed
        rendered = nf.nerf.render_image(
            results[name]["field"],
            results[name]["renderer"],
            batch["rays_o"],
            batch["rays_d"],
            background=test_dataset.background.to(device),
        )
    renders[name] = rendered["rgb"].clamp(0, 1).cpu()
    scores[name] = nf.psnr(renders[name], target)

ranked = sorted(scores, key=scores.get, reverse=True)
views = [("Ground truth", target.numpy())] + [
    (f"{name}\n{scores[name]:.2f} dB", renders[name].numpy()) for name in ranked
]
compare(views, "Absolute error")
plt.show()

## Orbit

The best field, rendered around the scene at the test views' elevation and
distance.

In [ ]:
best = summary["Model"].iloc[0]
poses = [
    nf.nerf.pose_spherical(theta, -30.0, test_dataset.radius)
    for theta in np.linspace(-180, 180, 60, endpoint=False)
]
with torch.compiler.set_stance("force_eager"):  # nor does an orbit
    frames = nf.nerf.render_views(
        results[best]["field"], results[best]["renderer"], test_dataset, poses
    )

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / f"{SCENE}_orbit.gif"
    imageio.mimsave(path, frames, fps=15, loop=0)
    print(best)
    display(IPyImage(filename=str(path)))